# Throw ball at face challenge
GitHub Repo for student's reference: https://tinyurl.com/LessonCode

In [ ]:
import time

import cv2
import numpy as np
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.1.190")
got.show_light_rgb([1, 2, 3], 255, 255, 255)
got.play_audio_tts("Connection successful")

In [ ]:
got.load_models(['color_recognition', 'face_recognition', 'word_recognition'])

In [ ]:
got.face_recognition_get_all_names()

## Stage 1: Ball pickup
Students are to show that they can detect, centralize, and approach the ball. Students can slightly adjust the ball position so that the arm can pick up the ball 

In [ ]:
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # -------------------------------------------------
            # Centralize logic
            # -------------------------------------------------
            if cx < 270:
                got.mecanum_translate_speed(-90, 20)
                action = "Moving Left"
            elif cx > 370:
                got.mecanum_translate_speed(90, 20)
                action = "Moving Right"
            else:
                got.mecanum_stop()
                action = "Stopped"

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
                f"Action: {action}"
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

In [ ]:
pick_up_cube = False
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # -------------------------------------------------
            # Approach logic
            # -------------------------------------------------
            if w < 180:
                got.mecanum_translate_speed(0, 20)
                action = "Approaching"
            else:
                got.mecanum_stop()
                action = "Stopped"
                pick_up_cube = True

            if pick_up_cube == True:
                got.mechanical_clamp_release()
                time.sleep(1)
                got.mechanical_joint_control(int((320 - cx) * 0.1), -50, -20, 1000)
                time.sleep(1)
                got.mechanical_clamp_close()
                time.sleep(1)
                got.mechanical_joint_control(0, 90, 90, 1000)
                pick_up_cube = False
                break

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
                f"Action: {action}"
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

## Stage 2: Moving into position
Students are to send instructions to the robot through text recognition to enable the robot to move into shooting position

In [ ]:
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        result = got.get_words_result()

        if result:
            # -------------------------------------------------
            # Text recognition and action decision logic
            # -------------------------------------------------
            if result == "LEFT":
                got.mecanum_turn_speed_times(2, 45, 90, 2)
            elif result == "RIGHT":
                got.mecanum_turn_speed_times(3, 45, 90, 0)
            elif result == "FORWARD":
                got.mecanum_move_speed(0, 20)
            elif result == "BACKWARD":
                got.mecanum_move_speed(1, 20)
        else:
            got.mecanum_stop()
            
        # Display the frame
        cv2.imshow("Text Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    cv2.destroyAllWindows()
    cv2.waitKey(1)

## Stage 3: Face Recognition and Shooting the ball at the face
Students are to recognize the correct face and shoot the ball at the correct person. Face needs to be knocked down to be considered a win.

Omit the face recognition part if there is not enough time

In [ ]:
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        result = got.get_face_recognition_total_info()

        for face in result:
            name, center_x, center_y, height, width, area = face

            # Calculate bounding-box corners
            left = int(center_x - width / 2)
            top = int(center_y - height / 2)
            right = int(center_x + width / 2)
            bottom = int(center_y + height / 2)
            
            # Draw bounding box
            cv2.rectangle(img, (left, top), (right, bottom), (0, 255, 0), 2 )
            
            # Draw person's name
            cv2.putText(img, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # Shoot at the face
            if name == "Coley":
                if center_x < 300:
                    got.mechanical_joint_control(30, 0, 0, 100)
                    got.mechanical_clamp_release()
                elif center_x > 340:
                    got.mechanical_joint_control(-30, 0, 0, 100)
                    got.mechanical_clamp_release()
                else:
                    got.mechanical_joint_control(0, 0, 0, 100)
                    got.mechanical_clamp_release()

            
        # Display the frame
        cv2.imshow("Face Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    cv2.destroyAllWindows()
    cv2.waitKey(1)